In [1]:
import random
from pymilvus import (
    connections, FieldSchema, CollectionSchema, DataType,
    Collection, utility
)

HOST = "localhost"
PORT = "19530"

COLLECTION_NAME = "demo_vectors"
DIM = 2 #128
TOPK = 5

#### Create a collection

In [2]:
# Connect to Milvus
connections.connect(alias="default", host=HOST, port=PORT)

# Drop collection if it exists (lab-friendly)
if utility.has_collection(COLLECTION_NAME):
    Collection(COLLECTION_NAME).drop()

# Define schema
pk = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False)
vec = FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=DIM)
txt = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=512)

schema = CollectionSchema(
    fields=[pk, vec, txt],
    description="Simple demo collection for vector search"
)

# Create collection
col = Collection(name=COLLECTION_NAME, schema=schema)

#### Create an index

In [3]:
# Create an index (IVF_FLAT is a friendly starter index)
index_params = {
    "index_type": "IVF_FLAT",
    "metric_type": "L2",
    "params": {"nlist": 128},
}
col.create_index(field_name="embedding", index_params=index_params)


Status(code=0, message=)

#### Insert a handful of vectors

In [4]:
# Insert some data
vector_count = 5 #200
ids = list(range(1, vector_count + 1))

vectors = []
vectors.append([0,1])
vectors.append([0,2])
vectors.append([0,3])
vectors.append([0,-1])
vectors.append([0,5])
#vectors = [make_vector(DIM) for _ in range(vector_count)]

# Create sample text for each vector.
texts = [f'sample item {i}' for i in ids]

col.insert([ids, vectors, texts])
col.flush()

#### Load the collection and search

In [5]:
# Load collection for search
col.load()

# Query vector (in a real app: embedding from a model)
#q = [make_vector(DIM)]
q = [[0,1]]

search_params = {"params": {"nprobe": 16}}
results = col.search(
    data=q,
    anns_field="embedding",
    param=search_params,
    limit=TOPK,
    output_fields=["text"],
)

print(f"\nTop {TOPK} results:")
for i, hit in enumerate(results[0], start=1):
    print(f"{i}. id={hit.id}  distance={hit.distance:.4f}  text={hit.entity.get('text')}")

# Optional: verify MinIO usage by checking bucket objects in MinIO Console.
connections.disconnect("default")


Top 5 results:
1. id=1  distance=0.0000  text=sample item 1
2. id=2  distance=1.0000  text=sample item 2
3. id=3  distance=4.0000  text=sample item 3
4. id=4  distance=4.0000  text=sample item 4
5. id=5  distance=16.0000  text=sample item 5
